# Notebook for running the winning model




In [ ]:
import json

import pandas as pd
from pathlib import Path
import processing.acled_events_processing as acled
from models.train_models import train_evaluate_model
from utils.data_prep import get_clean_combined_data

REPORTS_DIR = Path("evaluation/model_reports")
reports_dir = Path(REPORTS_DIR)
reports_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def summarise(label, subset):
    n_true_pos = subset["y_true"].sum()
    n_caught = subset[(subset["y_true"] == 1) & (subset["y_pred"] == 1)].shape[0]
    recall = n_caught / n_true_pos if n_true_pos else float("nan")
    n_pred_pos = subset["y_pred"].sum()
    precision = n_caught / n_pred_pos if n_pred_pos else float("nan")
    print(
        f"{label}: {len(subset)} rows, {n_true_pos} true escalations, "
        f"{n_caught} caught -> recall={recall:.3f}, precision={precision:.3f}"
    )
    return {
        "n_rows": len(subset),
        "n_true_pos": int(n_true_pos),
        "n_caught": n_caught,
        "recall": recall,
        "precision": precision,
    }

In [ ]:
def run_model(config, params):
    data_sources = [
        src
        for src, include in zip(
            ["food", "rain", "text"],
            [
                config["include_food"],
                config["include_rain"],
                config["include_text"],
            ],
        )
        if include
    ]

    model_data, predictor_cols = get_clean_combined_data(
        data_sources=data_sources,
        k=config["k"],
        event_col=config["event_col"],
        conflict_only_embeddings=config["conflict_only"],
    )

    final_params = {
        **params,
        "k": config["k"],
        "event_col": config["event_col"],
        "n_splits": config["n_splits"],
        "use_pca": config["use_pca"],
    }

    results, best_params, shap_importance, onset_predictions = train_evaluate_model(
        model_data,
        predictor_cols,
        final_params,
        best_params=True,  # skip RandomizedSearchCV
        use_pca=config["use_pca"],
        compute_shap=True,
        shap_sample_size=2000,
        return_onset_predictions=True,
    )
    return results, best_params, shap_importance, onset_predictions

In [ ]:
def model_report(label, config, params):
    """Runs a model and prints the full report, same as before, but also
    returns a flat dict of key metrics so multiple models can be compared
    in a table afterwards without re-typing numbers by hand.
    """
    results, best_params, shap_importance, onset_predictions = run_model(config, params)
    print("========MODEL REPORT========")
    print(f"---Model: {label}\n")
    print("---Results\n")
    print(results)
    print("---Best Params\n")
    print(best_params)
    print("---SHAP Importance\n")
    print(shap_importance)

    onset_predictions["year_month"] = onset_predictions["year_month"].astype(str)
    war_outbreak = "2023-04"
    print("Pre and post war:")
    pre_war = onset_predictions[onset_predictions["year_month"] < war_outbreak]
    post_war = onset_predictions[onset_predictions["year_month"] >= war_outbreak]

    pre_war_summary = summarise("Pre-war  (Jan-Mar 2023)", pre_war)
    post_war_summary = summarise("Post-war (Apr-Dec 2023)", post_war)

    key_regions = [
        "Khartoum",
        "North Darfur",
        "South Darfur",
        "West Darfur",
        "Central Darfur",
        "East Darfur",
        "West Kordofan",
        "South Kordofan",
    ]

    print("-----Key war-affected regions\n")
    key_region_rows = onset_predictions[onset_predictions["region"].isin(key_regions)]
    key_regions_summary = summarise("Key regions (all onset months)", key_region_rows)

    khartoum_rows = onset_predictions[onset_predictions["region"] == "Khartoum"]
    khartoum_summary = summarise("  Khartoum", khartoum_rows)

    for region in key_regions:
        if region == "Khartoum":
            continue
        region_rows = onset_predictions[onset_predictions["region"] == region]
        if region_rows["y_true"].sum() > 0:
            summarise(f"  {region}", region_rows)

    comparison_row = {
        "model": label,
        "onset_aupr": float(results["onset_aupr"]),
        "active_aupr": float(results["active_aupr"]),
        "pre_war_recall": pre_war_summary["recall"],
        "post_war_recall": post_war_summary["recall"],
        "khartoum_recall": khartoum_summary["recall"],
        "key_regions_recall": key_regions_summary["recall"],
    }

    return results, best_params, shap_importance, onset_predictions, comparison_row

In [ ]:
def save_model_report(label, results, best_params, shap_importance, onset_predictions):
    label_formatted = label.replace(" ", "_").replace("(", "").replace(")","")
    
    with open(REPORTS_DIR / f"{label_formatted}_results.json", "w") as f:
        json.dump(results, f, indent=2)
    
    with open(REPORTS_DIR / f"{label_formatted}_best_params.json", "w") as f:
        json.dump(best_params, f, indent=2)
        
    shap_importance.to_csv(REPORTS_DIR / f"{label_formatted}_shap.csv", index=False)
    onset_predictions.to_csv(REPORTS_DIR / f"{label_formatted}_onset_predictions.csv", index=False)

In [ ]:
def open_model_report(label):
    label_formatted = label.replace(" ", "_").replace("(", "").replace(")","")

    with open(REPORTS_DIR / f"{label_formatted}_results.json") as f:
            results = json.load(f)
    with open(REPORTS_DIR / f"{label_formatted}_best_params.json") as f:
        best_params = json.load(f)
    shap_importance = pd.read_csv(REPORTS_DIR / f"{label_formatted}_shap.csv")
    onset_predictions = pd.read_csv(REPORTS_DIR / f"{label_formatted}_onset_predictions.csv")
    
    return results, best_params, shap_importance, onset_predictions

## Model A - Structural only

In [ ]:
model_a_config = {
    "include_food": True,
    "include_rain": True,
    "include_text": False,
    "conflict_only": None,
    "k": 1.75,
    "event_col": "sub_event_type",
    "n_splits": 5,
    "use_pca": False,
}

# (acled_sub_food_rain_threshold_change_1.75_5, onset_aupr=0.3336, active_aupr=0.2025)
model_a_xgb_params = {
    "max_depth": 3,
    "min_child_weight": 1,
    "max_delta_step": 0,
    "gamma": 0,
    "learning_rate": 0.01,
    "subsample": 0.6,
    "colsample_bytree": 0.8,
    "reg_alpha": 2.0,
    "reg_lambda": 1,
    "colsample_bylevel": 1.0,
}

In [ ]:
results_a, best_params_a, shap_a, onset_preds_a, row_a = model_report(
    "Model A", model_a_config, model_a_xgb_params
)
save_model_report("Model A", results_a, best_params_a, shap_a, onset_preds_a)

## Model B

### Best model (conflict-only text)

In [ ]:
# --- Model B: numeric + food + rain + text (matched to Model A) ---
model_b_config = {
    "include_food": True,
    "include_rain": True,
    "include_text": True,
    "conflict_only": True,
    "k": 1.75,
    "event_col": "sub_event_type",
    "n_splits": 5,
    "use_pca": True,
}

# (acled_sub_food_rain_text_conflict_pca_1.75_5, onset_aupr=0.3941, active_aupr=0.2155)
model_b_xgb_params = {
    "max_depth": 7,
    "min_child_weight": 1,
    "max_delta_step": 0,
    "gamma": 3,
    "learning_rate": 0.01,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "reg_alpha": 2.0,
    "reg_lambda": 10,
    "colsample_bylevel": 0.8,
}

In [ ]:
(results_b_conflict,
    best_params_b_conflict,
    shap_b_conflict,
    onset_preds_b_conflict,
    row_b_conflict,
) = model_report("Model B (conflict-only text)", model_b_config, model_b_xgb_params)

save_model_report("Model B (conflict-only text)", results_b_conflict,
    best_params_b_conflict,
    shap_b_conflict,
    onset_preds_b_conflict,)

### Comparison (all text)

In [ ]:
# --- Model B: numeric + food + rain + text (matched to Model A) ---
model_b_all_config = {
    "include_food": True,
    "include_rain": True,
    "include_text": True,
    "conflict_only": False,
    "k": 1.75,
    "event_col": "sub_event_type",
    "n_splits": 5,
    "use_pca": True,
}

# (acled_sub_food_rain_text_conflict_pca_1.75_5, onset_aupr=0.3941, active_aupr=0.2155)
model_b_all_xgb_params = {
    "max_depth": 7,
    "min_child_weight": 1,
    "max_delta_step": 0,
    "gamma": 3,
    "learning_rate": 0.01,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "reg_alpha": 2.0,
    "reg_lambda": 10,
    "colsample_bylevel": 0.8,
}

In [ ]:
results_b_all, best_params_b_all, shap_b_all, onset_preds_b_all, row_b_all = (
    model_report("Model B (all-event text)", model_b_all_config, model_b_all_xgb_params)
)
save_model_report("Model B (all-event text)",results_b_all, best_params_b_all, shap_b_all, onset_preds_b_all)

In [ ]:
corpus_comparison = pd.DataFrame([row_b_conflict, row_b_all]).set_index("model")
corpus_comparison.loc["Difference (all-text - conflict-only)"] = (
    corpus_comparison.loc["Model B (all-event text)"]
    - corpus_comparison.loc["Model B (conflict-only text)"]
)
corpus_comparison.round(3)

In [ ]:
_, _, raw_df = acled.get_clean_data(k=1.75, event_col="sub_event_type")

if not isinstance(raw_df["year_month"].dtype, pd.PeriodDtype):
    raw_df["year_month"] = pd.to_datetime(raw_df["year_month"]).dt.to_period("M")

pre_war = raw_df[
    (raw_df["year_month"] >= "2023-01") & (raw_df["year_month"] <= "2023-03")
]

key_regions = [
    "Khartoum",
    "North Darfur",
    "South Darfur",
    "West Darfur",
    "Central Darfur",
    "East Darfur",
    "West Kordofan",
    "South Kordofan",
]

summary = (
    pre_war[pre_war["admin1"].isin(key_regions)]
    .groupby("admin1")
    .agg(total_events=("conflict", "size"), conflict_events=("conflict", "sum"))
)
summary["pct_conflict"] = (
    summary["conflict_events"] / summary["total_events"] * 100
).round(1)
summary.sort_values("conflict_events")

In [ ]:
import pandas as pd

from utils.constants import ACTIVE_END_DATE, ACTIVE_START_DATE

active_prevalence_records = []
for k_test in [1.75]:
    model_data, predictor_cols = get_clean_combined_data(
        data_sources=[],
        k=k_test,
        event_col="sub_event_type",
        conflict_only_embeddings=True,
    )
    active_slice = model_data[
        (model_data["year_month"] >= pd.Period(ACTIVE_START_DATE, freq="M"))
        & (model_data["year_month"] <= pd.Period(ACTIVE_END_DATE, freq="M"))
    ]
    n_total = len(active_slice)
    n_escalations = int(active_slice["target_escalation"].sum())
    active_prevalence_records.append(
        {
            "k": k_test,
            "active_prevalence_percentage": round(n_escalations / n_total * 100, 1),
            "n_escalations": n_escalations,
            "n_active_rows": n_total,
        }
    )

active_prevalence_df = pd.DataFrame(active_prevalence_records).set_index("k")
print(active_prevalence_df)

### AUPR relative to baseline

Raw AUPR values at k=1.75 are lower than at k=1.0 (Model A onset: 0.3379 vs 0.3213 previously;
active: 0.2242 vs 0.3693 previously), which could look like a regression. However, AUPR's
no-skill baseline equals the positive-class prevalence, which is itself lower at the stricter
k=1.75 threshold (22.7% onset, 16.2% active, vs 30.6% onset at k=1.0). Expressed as a lift over
that baseline, Model A achieves 1.49x on onset and 1.38x on active, a genuine, meaningful
improvement over chance, and the onset lift is comparable to or better than what was observed at
k=1.0. The apparent decline in raw AUPR is therefore largely explained by evaluating against a
stricter, more legitimate target, not by weaker model performance. [Model B's equivalent lift
figures are not yet available and require re-running with local embeddings access.]


### Regional performance

Onset-window recall varies substantially by region. Of the key regions in the onset months (Jan-Mar 2023), Model A caught 14 of the 27 true escalations. Notably, Model A only caught only 1 of 5 true escalations in Khartoum (recall 0.200) - the capital and the site of the actual April 2023 outbreak. Recall was better in East Darfur, catching 3 of 3 true esclations, and West Kordofan (4 of 4). 

Model B (which includes conflict-only text) has a similar regional performance in Khartoum but overall performance in key regions in onset months is poor at 6 of the 27 true escalations caught. 

The interesting thing at the regional level during the onset period is performance of Model B with all text. It manages to catch 21 of the 27 true escalations including a recall of 3/5 in Khartoum. Looking at the data for Khatoum this is likely because of the 220 events in the onset months, only 11.4% were conflict events, the others were things like strategic developments. 

These finding are of a very small sample size (1-5 true esclations per region) so can only be taken as tenative. Comparing Model B all text vs conflict-only text, all text 


However this pattern is tentative given very small per-region sample sizes (1-5 true escalations per region).
